# Autograd in PyTorch — Conceptual + Interview Notebook

### Goal
By the end of this notebook, you should be able to:

- Explain **what Autograd is and why PyTorch needs it**
- Understand **automatic differentiation** and the **computational graph**
- Use `requires_grad=True`
- Understand `.backward()` and `.grad`
- Connect Autograd with the **chain rule** and **backpropagation**
- Understand **gradient accumulation**
- Use `torch.no_grad()` and `.detach()` correctly
- Understand gradients for scalars, vectors, and tensors
- Connect Autograd to a real neural-network training loop
- Answer common **interview questions** confidently

> **Core mental model:**  
> `requires_grad=True` → build/track graph → `loss.backward()` → gradients → optimizer updates parameters


## 1. What is Autograd?

**Autograd** is PyTorch's automatic differentiation engine.

Its job is to automatically calculate derivatives (gradients) of a computation with respect to tensors that require gradients.

For example:

\[
y=x^2
\]

Then:

\[
\frac{dy}{dx}=2x
\]

At \(x=3\):

\[
\frac{dy}{dx}=6
\]

Instead of calculating this manually, PyTorch can do it for us.

### Interview definition

> **PyTorch Autograd is an automatic differentiation engine that tracks operations on tensors and automatically computes gradients through backpropagation.**

### Why do we need it?

Neural networks can have millions or billions of parameters. During training, we need gradients such as:

\[
\frac{\partial Loss}{\partial W_1},
\frac{\partial Loss}{\partial W_2},...
\]

Calculating all of them manually would be impractical. Autograd handles this automatically.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)


## 2. The Mathematical Foundation: Derivatives

Before using Autograd, remember what a gradient means.

Suppose:

\[
y=x^2
\]

The derivative tells us how much \(y\) changes when \(x\) changes:

\[
\frac{dy}{dx}=2x
\]

At \(x=3\):

\[
\frac{dy}{dx}=6
\]

Autograd automates this differentiation process.


In [ ]:
# Manual calculation vs Autograd

x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

print("x =", x)
print("y =", y)

# Calculate dy/dx
y.backward()

print("dy/dx =", x.grad)


### What happened?

When we created:

```python
x = torch.tensor(3.0, requires_grad=True)
```

we told PyTorch:

> "Track operations involving `x` because I may need its gradient."

Then:

```python
y = x ** 2
```

created a computation.

Finally:

```python
y.backward()
```

asked Autograd to compute:

\[
\frac{dy}{dx}
\]

The result is stored in:

```python
x.grad
```

So:

```text
requires_grad=True
        ↓
Track computation
        ↓
y = x²
        ↓
y.backward()
        ↓
x.grad = dy/dx
```


## 3. `requires_grad=True`

This is one of the most important Autograd concepts.

```python
x = torch.tensor(3.0, requires_grad=True)
```

means PyTorch should track operations involving `x` for automatic differentiation.

Compare:


In [ ]:
x1 = torch.tensor(3.0)
x2 = torch.tensor(3.0, requires_grad=True)

print("x1 requires grad:", x1.requires_grad)
print("x2 requires grad:", x2.requires_grad)


### Important distinction

`requires_grad=True` does **not** mean "calculate the gradient immediately."

It means:

> **Track the operations so that a gradient can be calculated later.**

The gradient is normally computed when you call:

```python
.backward()
```


## 4. Computational Graph

PyTorch Autograd tracks operations and forms a **dynamic computational graph**.

Consider:

\[
a=x^2
\]

\[
b=a+3
\]

\[
y=4b
\]

The conceptual graph is:

```text
       x
       │
       ▼
     x²
       │
       ▼
    a + 3
       │
       ▼
     × 4
       │
       ▼
       y
```

PyTorch remembers enough information about these operations to move backward through the graph and calculate derivatives.

### Why "dynamic"?

The graph is built as your Python code executes. Different executions can create different computation paths.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

a = x ** 2
b = a + 3
y = b * 4

print("x =", x)
print("a =", a)
print("b =", b)
print("y =", y)

y.backward()

print("\nGradient dy/dx =", x.grad)


### Verify the gradient manually

We have:

\[
y=4(x^2+3)
\]

Therefore:

\[
y=4x^2+12
\]

and:

\[
\frac{dy}{dx}=8x
\]

At \(x=2\):

\[
\frac{dy}{dx}=16
\]

Autograd gives the same result.

This is the key connection:

> **Autograd is implementing differentiation of the computational graph.**


## 5. `.backward()` — What does it actually do?

When you call:

```python
loss.backward()
```

PyTorch performs reverse-mode automatic differentiation through the computation graph.

For a neural network, this is the mathematical basis of **backpropagation**.

Conceptually:

```text
Forward pass
Input → Layer 1 → Layer 2 → Prediction → Loss

Backward pass
Input ← Layer 1 ← Layer 2 ← Prediction ← Loss
          gradients flow backward
```

The backward pass computes derivatives of the loss with respect to trainable parameters.


## 6. Autograd and the Chain Rule

The chain rule is the mathematical foundation of backpropagation.

Suppose:

\[
y=f(g(x))
\]

Then:

\[
\frac{dy}{dx}
=
\frac{dy}{dg}
\frac{dg}{dx}
\]

### Example

\[
u=x^2
\]

\[
y=3u+1
\]

Then:

\[
\frac{du}{dx}=2x
\]

and:

\[
\frac{dy}{du}=3
\]

Therefore:

\[
\frac{dy}{dx}
=
\frac{dy}{du}\frac{du}{dx}
=
3(2x)=6x
\]

At \(x=2\):

\[
\frac{dy}{dx}=12
\]

Autograd applies this idea across potentially very large computational graphs.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

u = x ** 2
y = 3 * u + 1

y.backward()

print("Autograd gradient:", x.grad)
print("Expected gradient:", 6 * 2)


## 7. `.grad` — Where are gradients stored?

After a successful backward pass, a leaf tensor that requires gradients normally has its gradient available through:

```python
tensor.grad
```

Example:

```python
x.grad
```

For a scalar example:

\[
y=x^3
\]

\[
\frac{dy}{dx}=3x^2
\]

At \(x=4\), the gradient is 48.


In [ ]:
x = torch.tensor(4.0, requires_grad=True)

y = x ** 3
y.backward()

print("x:", x)
print("y:", y)
print("gradient:", x.grad)


## 8. Leaf Tensors vs Non-Leaf Tensors

This is an important interview topic.

A **leaf tensor** is generally a tensor created directly by the user that is not the result of an Autograd operation.

Example:

```python
x = torch.tensor(2.0, requires_grad=True)
```

`x` is a leaf tensor.

But:

```python
y = x ** 2
```

makes `y` a non-leaf result of an operation.

Check it:


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

print("x.is_leaf:", x.is_leaf)
print("y.is_leaf:", y.is_leaf)

print("x.grad_fn:", x.grad_fn)
print("y.grad_fn:", y.grad_fn)


### `grad_fn`

A non-leaf tensor often has a `grad_fn` describing the backward operation that created it.

For example, after:

```python
y = x ** 2
```

PyTorch knows that `y` was produced by a power operation.

This information helps Autograd traverse the graph during backpropagation.

### Interview point

> Leaf tensors that require gradients are where PyTorch normally accumulates `.grad` by default.


## 9. Why Gradients Accumulate

PyTorch **accumulates gradients by default**.

This is extremely important when writing training loops.

Example:

\[
y=x^2
\]

At \(x=2\):

\[
\frac{dy}{dx}=4
\]

If we perform another backward pass without clearing the old gradient, the new gradient is added to the existing one.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()

print("After first backward:", x.grad)

y2 = x ** 2
y2.backward()

print("After second backward:", x.grad)


The second gradient is added:

\[
4+4=8
\]

This is why training loops normally contain:

```python
optimizer.zero_grad()
```

before a new backward pass.

### Important interview answer

> **PyTorch accumulates gradients because `.backward()` adds gradients into the existing `.grad` buffers. Therefore, gradients must usually be cleared between training iterations.**


## 10. `zero_grad()` and the Training Loop

A simplified training step is:

```python
optimizer.zero_grad()
output = model(x)
loss = criterion(output, y)
loss.backward()
optimizer.step()
```

### Meaning

| Step | Purpose |
|---|---|
| `zero_grad()` | Clear old gradients |
| `model(x)` | Forward pass |
| `criterion(...)` | Calculate loss |
| `loss.backward()` | Compute gradients |
| `optimizer.step()` | Update parameters |

The important relationship is:

```text
Forward
   ↓
Loss
   ↓
Backward / Autograd
   ↓
Gradients
   ↓
Optimizer
   ↓
Updated weights
```


## 11. A Simple Neural Network Example

Now let's see Autograd inside a real neural-network training loop.

We will create a tiny regression problem:

\[
y=2x+1
\]

The model will learn the weight and bias.


In [ ]:
import torch
import torch.nn as nn

# Training data
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])

# Simple linear model
model = nn.Linear(1, 1)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print("Initial weight:", model.weight.item())
print("Initial bias:", model.bias.item())


In [ ]:
for epoch in range(1000):
    # 1. Clear previous gradients
    optimizer.zero_grad()

    # 2. Forward pass
    prediction = model(x)

    # 3. Calculate loss
    loss = criterion(prediction, y)

    # 4. Backward pass: Autograd calculates gradients
    loss.backward()

    # 5. Update parameters
    optimizer.step()

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1:4d} | Loss: {loss.item():.6f}")

print("\nLearned weight:", model.weight.item())
print("Learned bias:", model.bias.item())


### Where is Autograd in this code?

The critical line is:

```python
loss.backward()
```

Autograd calculates:

\[
\frac{\partial Loss}{\partial Weight}
\]

and:

\[
\frac{\partial Loss}{\partial Bias}
\]

Then:

```python
optimizer.step()
```

uses those gradients to update the parameters.

So **Autograd computes gradients; the optimizer uses them.**

> Autograd does **not** update weights by itself.


## 12. Inspecting Parameter Gradients

Let's explicitly inspect the gradients generated by Autograd.



In [ ]:
model = nn.Linear(1, 1)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

optimizer.zero_grad()

prediction = model(x)
loss = criterion(prediction, y)

loss.backward()

print("Weight gradient:")
print(model.weight.grad)

print("\nBias gradient:")
print(model.bias.grad)


The `.grad` values are exactly what the optimizer needs.

For example, gradient descent conceptually uses:

\[
W_{new}=W_{old}-\eta\frac{\partial L}{\partial W}
\]

where:

- \(W\) = parameter
- \(L\) = loss
- \(\eta\) = learning rate

So the complete learning mechanism is:

```text
Autograd
   ↓
calculate ∂Loss/∂W
   ↓
Optimizer
   ↓
update W
```


## 13. `torch.no_grad()`

During inference/evaluation, we usually do not need gradients.

Use:

```python
with torch.no_grad():
    output = model(x)
```

This tells PyTorch not to track operations for gradient computation in that block.

### Why use it?

- Reduces memory usage
- Avoids unnecessary graph construction
- Can improve inference efficiency

Example:


In [ ]:
model.eval()

with torch.no_grad():
    prediction = model(x)

print(prediction)


### `model.eval()` vs `torch.no_grad()`

These are **not the same thing**.

`model.eval()`:

> Changes the behavior of layers such as Dropout and BatchNorm during evaluation.

`torch.no_grad()`:

> Disables gradient tracking for the enclosed operations.

They are often used together:

```python
model.eval()

with torch.no_grad():
    output = model(x)
```


## 14. `detach()`

`detach()` creates a tensor that is disconnected from the current Autograd graph.

Example:

```python
y = x.detach()
```

The detached tensor does not continue gradient tracking through the original computation.

This is useful when you want to use a tensor's value without allowing gradients to flow through that part of the graph.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
z = y.detach()

print("x requires_grad:", x.requires_grad)
print("y requires_grad:", y.requires_grad)
print("z requires_grad:", z.requires_grad)


### `detach()` vs `no_grad()`

**`torch.no_grad()`** is a context manager that prevents tracking within a block:

```python
with torch.no_grad():
    y = model(x)
```

**`detach()`** works on a particular tensor:

```python
y = output.detach()
```

A useful mental model:

```text
no_grad() → "Don't track operations in this block."

detach()  → "Disconnect this tensor from its graph."


## 15. Scalar Outputs and `.backward()`

The simplest use of:

```python
loss.backward()
```

is when `loss` is a scalar.

For example:

```python
loss.shape == torch.Size([])
```

This is common in neural-network training because loss functions such as mean squared error typically return a scalar loss.



In [ ]:
x = torch.tensor([2.0, 3.0], requires_grad=True)

y = (x ** 2).sum()

print("y:", y)
print("y shape:", y.shape)

y.backward()

print("x.grad:", x.grad)


Here:

\[
y=2^2+3^2=13
\]

The gradient is:

\[
\frac{\partial y}{\partial x_1}=2x_1=4
\]

\[
\frac{\partial y}{\partial x_2}=2x_2=6
\]

So:

```text
x.grad = [4, 6]
```


## 16. Non-Scalar Outputs

A common interview question is:

> Can you directly call `.backward()` on a non-scalar tensor?

Usually, **no**. For a vector/tensor output, PyTorch needs an appropriate gradient argument (often called the upstream gradient).

Example:


In [ ]:
x = torch.tensor([2.0, 3.0], requires_grad=True)

y = x ** 2

# y is not a scalar
print("y:", y)
print("y shape:", y.shape)

# Provide an upstream gradient
y.backward(torch.ones_like(y))

print("x.grad:", x.grad)


This computes the gradient of the scalar quantity:

\[
[1,1]\cdot y
\]

which is:

\[
y_1+y_2
\]

The important interview idea is:

> **For a scalar output, `.backward()` can be called directly. For a non-scalar output, supply the gradient/upstream vector (or reduce the output to a scalar).**


## 17. `grad_fn` and the Backward Graph

Let's inspect the graph-related information.



In [ ]:
x = torch.tensor(2.0, requires_grad=True)

a = x * 3
b = a + 4
y = b ** 2

print("x.grad_fn:", x.grad_fn)
print("a.grad_fn:", a.grad_fn)
print("b.grad_fn:", b.grad_fn)
print("y.grad_fn:", y.grad_fn)


For a leaf tensor such as `x`, `grad_fn` is normally `None`.

For results created by operations, `grad_fn` identifies the backward operation associated with that result.

This is part of how Autograd knows how to propagate gradients backward.


## 18. `retain_graph=True`

Normally, after a backward pass, PyTorch frees parts of the computation graph that are no longer needed.

If you need to perform another backward pass through the same graph, you may need:

```python
y.backward(retain_graph=True)
```

Example:


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2

y.backward(retain_graph=True)
print("First backward:", x.grad)

# Clear the gradient so we can see the second result separately
x.grad.zero_()

y.backward()
print("Second backward:", x.grad)


### Interview warning

Do not use `retain_graph=True` automatically.

It keeps the graph alive and can increase memory usage.

Use it only when you genuinely need to perform another backward pass through the same graph.


## 19. Gradient Flow Through a Simple Neural Network

Consider a neuron:

\[
z=wx+b
\]

\[
a=ReLU(z)
\]

\[
L=(a-y)^2
\]

During backpropagation, Autograd applies the chain rule:

\[
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial a}
\frac{\partial a}{\partial z}
\frac{\partial z}{\partial w}
\]

You do not manually calculate every intermediate derivative in PyTorch. Autograd tracks the operations and performs the backward computation.


In [ ]:
import torch
import torch.nn as nn

x = torch.tensor(2.0)
target = torch.tensor(5.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

z = w * x + b
a = torch.relu(z)
loss = (a - target) ** 2

loss.backward()

print("z =", z.item())
print("a =", a.item())
print("loss =", loss.item())

print("\n∂Loss/∂w =", w.grad.item())
print("∂Loss/∂b =", b.grad.item())


## 20. A Visual Training Concept

A neural network learns through this cycle:

```text
             FORWARD PASS
Input ─────────────────────────→ Prediction
                                  │
                                  ▼
                                Loss
                                  │
                                  │ backward()
                                  ▼
             BACKWARD PASS
Input ←──────────────────────── Gradients

                                  │
                                  ▼
                            Optimizer
                                  │
                                  ▼
                           New parameters
                                  │
                                  └────→ next forward pass
```

### Three different responsibilities

**Autograd**
- Builds/tracks the graph
- Calculates gradients

**Loss function**
- Measures prediction error

**Optimizer**
- Uses gradients to update parameters

Do not confuse these roles.


## 21. Common Mistakes

### Mistake 1 — Forgetting `requires_grad=True`

```python
x = torch.tensor(2.0)
y = x ** 2
y.backward()
```

This cannot calculate a gradient for `x` because `x` was not marked for gradient tracking.

---

### Mistake 2 — Forgetting `zero_grad()`

Gradients accumulate.

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

---

### Mistake 3 — Thinking `backward()` updates weights

It does not.

```python
loss.backward()
```

calculates gradients.

```python
optimizer.step()
```

updates parameters.

---

### Mistake 4 — Using gradients during inference

Use:

```python
with torch.no_grad():
    output = model(x)
```

when gradients are not needed.

---

### Mistake 5 — Confusing `eval()` with `no_grad()`

They solve different problems:

```text
model.eval()       → evaluation behavior of certain layers
torch.no_grad()    → gradient tracking
```


# 22. Interview Questions & Answers

### Q1. What is Autograd in PyTorch?

**Answer:**

> Autograd is PyTorch's automatic differentiation engine. It tracks operations on tensors and automatically computes gradients through backpropagation.

---

### Q2. What does `requires_grad=True` mean?

> It tells PyTorch to track operations involving that tensor so that gradients can later be calculated with respect to it.

---

### Q3. What does `.backward()` do?

> It performs reverse-mode automatic differentiation through the computational graph and accumulates gradients into the `.grad` attributes of relevant leaf tensors.

---

### Q4. Where are gradients stored?

> For leaf tensors that require gradients, gradients are normally stored in the tensor's `.grad` attribute.

---

### Q5. Why do we use `optimizer.zero_grad()`?

> PyTorch accumulates gradients by default, so `zero_grad()` clears the gradients from the previous iteration before calculating new ones.

---

### Q6. Does `.backward()` update model parameters?

> No. `.backward()` calculates gradients. The optimizer, through `optimizer.step()`, updates the parameters.

---

### Q7. What is a computational graph?

> It is a graph representing the operations used to compute an output from input tensors. Autograd uses this graph to calculate derivatives during the backward pass.

---

### Q8. What is the relationship between Autograd and backpropagation?

> Backpropagation is the algorithmic process of propagating gradients backward through a computation graph. PyTorch Autograd automates the differentiation required for this process.

---

### Q9. Why use `torch.no_grad()`?

> To disable gradient tracking when gradients are not needed, such as during inference, reducing unnecessary computation and memory usage.

---

### Q10. Difference between `detach()` and `no_grad()`?

> `no_grad()` disables gradient tracking for operations inside a context, while `detach()` creates a tensor disconnected from the current computation graph.

---

### Q11. What is `grad_fn`?

> It identifies the backward operation associated with a tensor produced by an Autograd-tracked operation.

---

### Q12. What is a leaf tensor?

> A leaf tensor is generally a tensor created directly by the user rather than as the result of an Autograd operation. Trainable parameters are typically leaf tensors.

---

### Q13. Why can gradients accumulate?

> PyTorch's backward pass adds gradients to existing `.grad` buffers instead of replacing them, which is why gradients normally need to be cleared between iterations.

---

### Q14. What is the mathematical foundation of backpropagation?

> The chain rule of calculus.

---

### Q15. Can `.backward()` be called directly on a vector?

> A backward call without an explicit upstream gradient is intended for scalar outputs. For a non-scalar output, provide an appropriate gradient argument or reduce the output to a scalar.


# 23. Mini Practical Challenge

Try to solve these without looking at the answer.

### Challenge 1

Calculate the gradient of:

\[
y=5x^2+2x+1
\]

at \(x=3\) using PyTorch Autograd.

Expected mathematical result:

\[
\frac{dy}{dx}=10x+2
\]

---

### Challenge 2

Create two tensors:

```python
x = ...
w = ...
```

with gradients enabled and calculate:

\[
y=wx
\]

Then print:

- `x.grad`
- `w.grad`

---

### Challenge 3

Create a small linear model and show:

1. Forward pass
2. Loss
3. `loss.backward()`
4. Weight gradient
5. `optimizer.step()`

---

### Challenge 4

Explain in your own words:

> Why does a neural network need Autograd?

A strong answer should mention:

**loss → derivatives/gradients → chain rule → parameter updates**


In [ ]:
# Challenge 1 — Solution

x = torch.tensor(3.0, requires_grad=True)

y = 5 * x**2 + 2 * x + 1

y.backward()

print("y =", y.item())
print("dy/dx =", x.grad.item())
print("Expected =", 10 * 3 + 2)


# 24. Final Cheat Sheet

| Concept | Remember |
|---|---|
| Autograd | Automatic differentiation engine |
| `requires_grad=True` | Track operations for gradients |
| Computational graph | Records tracked operations |
| `.backward()` | Computes/propagates gradients |
| `.grad` | Stores accumulated gradient |
| `zero_grad()` | Clears old gradients |
| `grad_fn` | Backward operation associated with a result |
| `detach()` | Disconnect tensor from graph |
| `torch.no_grad()` | Disable gradient tracking in a block |
| `retain_graph=True` | Keep graph for another backward pass |
| Chain rule | Mathematical foundation of backpropagation |
| `optimizer.step()` | Updates model parameters |

## The most important flow

```text
                 PyTorch Training

Input
  ↓
Model
  ↓
Prediction
  ↓
Loss
  ↓
loss.backward()
  ↓
Autograd
  ↓
Gradients
  ↓
optimizer.step()
  ↓
Updated Parameters
```

### One-sentence interview answer

> **Autograd is PyTorch's automatic differentiation engine that dynamically tracks tensor operations and uses the resulting computation graph to calculate gradients during backpropagation, which optimizers then use to update model parameters.**


# 25. Suggested Learning Order After Autograd

Once you are comfortable with this notebook, study these topics in order:

1. **Tensor operations**
2. **Computational graphs**
3. **Gradient descent**
4. **Backpropagation mathematically**
5. **Loss functions**
6. **Optimizers — SGD, Momentum, Adam**
7. **PyTorch `nn.Module`**
8. **Dataset & DataLoader**
9. **Training / validation loops**
10. **CNN training**
11. **Learning rate scheduling**
12. **Gradient clipping**
13. **Mixed precision**
14. **Transfer learning**

For deep learning interviews, the most important connection to remember is:

\[
oxed{
Forward\ Pass
ightarrow
Loss
ightarrow
Backpropagation
ightarrow
Autograd
ightarrow
Gradients
ightarrow
Optimizer
ightarrow
Updated\ Weights
}
\]
